# LlamaIndex function and query-engine tools

An original, deterministic LlamaIndex companion lesson.

## Learning objectives

- Define metadata for function and query-engine tool styles.
- Validate tool names and arguments before dispatch.
- Produce deterministic multiplication and weather results.

## Setup

The offline registry models LlamaIndex tool boundaries using ordinary Python. Validation is explicit so malformed agent-shaped calls fail before application functions run.


The optional live example uses `python -m pip install llama-index llama-index-llms-huggingface-api`. Optional Phoenix tracing uses `python -m pip install arize-phoenix-otel`.

## Complete implementation

Run the next cell to build the complete offline demonstration.

In [1]:
def multiply(a, b):
    if type(a) is not int or type(b) is not int:
        raise ValueError("multiply requires integer arguments")
    return a * b

class WeatherQueryEngine:
    def __init__(self, records):
        self.records = dict(records)

    def query(self, city):
        normalized_city = city.strip().casefold()
        if normalized_city not in self.records:
            return "unknown"
        return self.records[normalized_city]

weather_query_engine = WeatherQueryEngine({"riga": "clear", "oslo": "rain"})
tool_catalog = {
    "multiply": {
        "kind": "function",
        "description": "Multiply two integers.",
        "required_arguments": {"a", "b"},
        "callable": multiply,
    },
    "weather": {
        "kind": "query-engine",
        "description": "Query an embedded weather record by city.",
        "required_arguments": {"city"},
        "callable": weather_query_engine.query,
    },
}

tool_calls = [
    {"name": "multiply", "arguments": {"a": 6, "b": 7}},
    {"name": "weather", "arguments": {"city": "Riga"}},
]

def validate_and_dispatch(call, catalog):
    tool_name = call.get("name")
    if tool_name not in catalog:
        raise ValueError("unknown tool")
    tool = catalog[tool_name]
    arguments = call.get("arguments", {})
    if set(arguments) != tool["required_arguments"]:
        raise ValueError("arguments do not match tool metadata")
    return tool["callable"](**arguments)

tool_results = {
    call["name"]: validate_and_dispatch(call, tool_catalog)
    for call in tool_calls
}

## Deterministic assertions

Run these checks to verify the lesson without credentials, network access, or third-party packages.

In [2]:
assert tool_catalog["multiply"]["kind"] == "function"
assert tool_catalog["weather"]["kind"] == "query-engine"
assert set(tool_catalog) == {"multiply", "weather"}
assert tool_results == {"multiply": 42, "weather": "clear"}

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

A function tool exposes a callable directly, while a query-engine tool adapts a richer query interface. Both require accurate descriptions and argument contracts because agent selection is only safe when dispatch validates the boundary.

## Credentials

Offline tools need no credentials. The optional live cell reads `HF_TOKEN` only after checking that it exists and never prints it. Phoenix is separately gated by `PHOENIX_API_KEY`. A Gmail tool would additionally require Google OAuth consent and scoped tokens, which are described here but not requested or stored.

In [3]:
import os

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional LlamaIndex example.")
else:
    try:
        from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
        from llama_index.core import Document, SummaryIndex
        from llama_index.core.tools import FunctionTool, QueryEngineTool, ToolMetadata
    except ImportError:
        print("SKIPPED: llama-index is not installed; run `python -m pip install llama-index llama-index-llms-huggingface-api`.")
    else:
        phoenix_api_key = os.getenv("PHOENIX_API_KEY")
        if phoenix_api_key:
            try:
                from phoenix.otel import register
            except ImportError:
                print("SKIPPED: Phoenix tracing is not installed; run `python -m pip install arize-phoenix-otel`.")
            else:
                tracer_provider = register(
                    project_name="hf-agents-course-llama-index",
                    endpoint="https://app.phoenix.arize.com/v1/traces",
                    headers={"api_key": phoenix_api_key},
                    auto_instrument=True,
                )
                print("Phoenix tracing configured.")
        else:
            print("SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.")

        llm = HuggingFaceInferenceAPI(
            model_name="Qwen/Qwen2.5-7B-Instruct",
            token=hf_token,
            task="text-generation",
        )
        def live_multiply(a: int, b: int) -> int:
            """Multiply two integers."""
            return a * b

        function_tool = FunctionTool.from_defaults(fn=live_multiply)
        weather_index = SummaryIndex.from_documents([Document(text="Riga weather is clear.")])
        weather_engine = weather_index.as_query_engine(llm=llm)
        query_engine_tool = QueryEngineTool(
            query_engine=weather_engine,
            metadata=ToolMetadata(
                name="weather",
                description="Answer questions about the embedded Riga weather record.",
            ),
        )
        print("Configured one FunctionTool and one QueryEngineTool for a hosted-HF agent.")

SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.


Configured one FunctionTool and one QueryEngineTool for a hosted-HF agent.


## Completion checklist

- [ ] Compared function and query-engine metadata.
- [ ] Validated both structured calls.
- [ ] Verified multiplication and weather outputs.

## Attribution

The validation and dispatch example is original and informed by the official [LlamaIndex tools documentation](https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/tools/). No upstream notebook implementation is copied. Notebook code carries the Apache-2.0 metadata license.